# Enterprise RAG Quality Evaluation Pipeline

An interactive, modular Jupyter Notebook for testing, benchmarking, and tracking the retrieval and generation quality of the LangGraph documentation chatbot.

All configurable parameters are centralized in `eval_config.json` (mirroring `crawler_config.json`).

### Pipeline Stages:
1. **Setup & Config Loading**: Load `eval_config.json`, verify environment, and configure OpenRouter RAGAS wrappers.
2. **Step 1: Testset Generation (Optional)**: Synthesize multi-hop evaluation questions from `0.doc.json` via Knowledge Graph -> `1.dataset.json`.
3. **Step 2: Query LangGraph Agent**: Execute evaluation queries directly via in-process `agent_graph.ainvoke()` (or via LangGraph Server HTTP API).
4. **Step 3: Compute RAGAS Metrics**: Score Faithfulness, Answer Relevancy, Context Precision, and Context Recall -> `2.run_eval_*.csv`.
5. **Step 4: Benchmark Evolution, Visualization & Journal Sync**: Sync latest run to `JOURNAL.md`, inspect progression table, and plot metric trajectories.

*(Note: LangSmith dataset ingestion is decoupled into standalone script `upload_langsmith.py`)*

In [ ]:
# region Setup
import asyncio
import glob
import json
import os
import re
import sys
import time
from datetime import datetime
from pathlib import Path

import httpx
import matplotlib.pyplot as plt
import pandas as pd
from datasets import Dataset
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas import evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    answer_relevancy,
    context_precision,
    context_recall,
    faithfulness,
)
from ragas.run_config import RunConfig

# Path resolution: ensure project root is in sys.path
CURRENT_DIR = Path.cwd() if Path.cwd().name == "eval" else Path.cwd() / "dataset" / "eval"
ROOT_DIR = CURRENT_DIR.parents[1]
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

load_dotenv(ROOT_DIR / ".env")

# Centralized configuration loading (mirroring crawler_config.json style)
CONFIG_FILE = CURRENT_DIR / "eval_config.json"
if CONFIG_FILE.exists():
    EVAL_CONFIG = json.loads(CONFIG_FILE.read_text(encoding="utf-8"))
    print(f"[Setup-init] Loaded configuration from {CONFIG_FILE.name}")
else:
    EVAL_CONFIG = {}
    print(f"[Setup-init] Warning: {CONFIG_FILE.name} not found, using internal defaults.")

DOCS_FILE = CURRENT_DIR / EVAL_CONFIG.get("generator", {}).get("doc_path", "0.doc.json")
DATASET_FILE = CURRENT_DIR / EVAL_CONFIG.get("query", {}).get("dataset_path", "1.dataset.json")

print(f"[Setup-init] Current Directory: {CURRENT_DIR}")
print(f"[Setup-init] Root Directory:    {ROOT_DIR}")
print(f"[Setup-init] Document Source:   {DOCS_FILE.name} ({'Found' if DOCS_FILE.exists() else 'Missing'})")
print(f"[Setup-init] Dataset Target:    {DATASET_FILE.name} ({'Found' if DATASET_FILE.exists() else 'Missing'})")
print(f"[Setup-init] OpenRouter Key:    {'Set' if os.getenv('OPENROUTER_API_KEY') else 'Missing'}")


def get_eval_models(temperature: float = 0.0, is_generator: bool = False):
    """Initializes LLM and Embeddings wrappers for Ragas evaluation and testset generation."""
    api_key = os.getenv("OPENROUTER_API_KEY")
    if not api_key:
        raise ValueError("[EvalClient-init] OPENROUTER_API_KEY is not set in environment.")

    base_url = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
    model = os.getenv("OPENROUTER_MODEL", "deepseek/deepseek-v4-flash-0731")
    embed_model = os.getenv("OPENROUTER_EMBED_MODEL", "nvidia/nemotron-3-embed-1b:free")
    provider_sort = os.getenv("OPENROUTER_PROVIDER_SORT", "throughput")
    provider_ignore_raw = os.getenv("OPENROUTER_PROVIDER_IGNORE", "wafer")
    provider_ignore = [p.strip() for p in provider_ignore_raw.split(",") if p.strip()]

    provider_config = {"allow_fallbacks": True}
    if provider_sort:
        provider_config["sort"] = provider_sort
    if provider_ignore:
        provider_config["ignore"] = provider_ignore

    extra_body = {
        "provider": provider_config,
        "reasoning": {
            "effort": "medium",
        },
    }

    llm = ChatOpenAI(
        model=model,
        api_key=api_key,
        base_url=base_url,
        temperature=temperature,
        max_tokens=16384,
        request_timeout=240.0,
        extra_body=extra_body,
    )
    embeddings = OpenAIEmbeddings(
        model=embed_model,
        api_key=api_key,
        base_url=base_url,
        check_embedding_ctx_length=False,
        model_kwargs={"encoding_format": "float"},
    )

    llm_wrapper = (
        LangchainLLMWrapper(llm, is_finished_parser=lambda _: True)
        if is_generator
        else LangchainLLMWrapper(llm)
    )
    return llm_wrapper, LangchainEmbeddingsWrapper(embeddings)
# endregion

## Step 1: Synthetic Test Dataset Generation (Optional)

Generate diverse, multi-hop question-answer test sets directly from `0.doc.json` using Ragas `TestsetGenerator` and Knowledge Graph extraction.
Configuration is derived from `eval_config.json["generator"]`.

> **Note**: This step makes calls to OpenRouter. If you already have `1.dataset.json`, you can skip directly to Step 2.

In [ ]:
# region Generator
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document
from ragas.testset import TestsetGenerator
from ragas.testset.graph import KnowledgeGraph
from ragas.testset.synthesizers.multi_hop import (
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
)
from ragas.testset.synthesizers.single_hop.specific import (
    SingleHopSpecificQuerySynthesizer,
)
from ragas.testset.transforms.engine import Parallel
from ragas.testset.transforms.extractors import EmbeddingExtractor, SummaryExtractor
from ragas.testset.transforms.extractors.llm_based import NERExtractor, ThemesExtractor
from ragas.testset.transforms.filters import CustomNodeFilter
from ragas.testset.transforms.relationship_builders import (
    CosineSimilarityBuilder,
    OverlapScoreBuilder,
)


def load_input_documents(doc_path: Path, max_chunks: int = 20) -> list[Document]:
    """Loads documents from text files or sampled JSON chunks."""
    if not doc_path.exists():
        raise FileNotFoundError(f"[DatasetGenerator-load] Target document not found: {doc_path}")

    if doc_path.suffix.lower() == ".json":
        items = json.loads(doc_path.read_text(encoding="utf-8"))
        items = items if isinstance(items, list) else [items]
        docs = []
        for item in items:
            meta = item.get("metadata", {})
            content = (
                meta.get("big") or meta.get("summary") or item.get("small") or ""
            ).strip()
            if len(content) > 50:
                docs.append(Document(page_content=content, metadata={"id": item.get("id", "")}))
            if 0 < max_chunks <= len(docs):
                break
        return docs

    return TextLoader(str(doc_path), encoding="utf-8").load()


def build_query_distribution(ragas_llm, weights: dict[str, float]) -> list[tuple]:
    """Builds a normalized 4-path query distribution matrix."""
    synthesizers = {
        "single_specific": SingleHopSpecificQuerySynthesizer(
            llm=ragas_llm, property_name="entities", name="single_hop_specific"
        ),
        "single_abstract": SingleHopSpecificQuerySynthesizer(
            llm=ragas_llm, property_name="themes", name="single_hop_abstract"
        ),
        "multi_specific": MultiHopSpecificQuerySynthesizer(
            llm=ragas_llm, name="multi_hop_specific"
        ),
        "multi_abstract": MultiHopAbstractQuerySynthesizer(
            llm=ragas_llm, name="multi_hop_abstract"
        ),
    }
    raw = [(synthesizers[k], max(0.0, weights.get(k, 0.25))) for k in synthesizers]
    total = sum(w for _, w in raw) or 1.0
    return [(s, w / total) for s, w in raw if w > 0]


def generate_eval_dataset_from_docs(
    doc_path: Path,
    output_path: Path,
    test_size: int = 30,
    max_chunks: int = 20,
    weights: dict[str, float] | None = None,
) -> list[dict]:
    """Synthesizes evaluation test samples into clean Ragas schema (question, ground_truth, ground_truth_contexts)."""
    print(f"[DatasetGenerator-load] Loading up to {max_chunks} chunks from: {doc_path.name}")
    docs = load_input_documents(doc_path, max_chunks=max_chunks)
    if not docs:
        raise ValueError(f"[DatasetGenerator-load] No readable document content found in {doc_path}")

    ragas_llm, ragas_embeddings = get_eval_models(is_generator=True)
    transforms = [
        SummaryExtractor(llm=ragas_llm),
        CustomNodeFilter(llm=ragas_llm),
        Parallel(
            EmbeddingExtractor(
                embedding_model=ragas_embeddings,
                property_name="summary_embedding",
                embed_property_name="summary",
            ),
            ThemesExtractor(llm=ragas_llm),
            NERExtractor(llm=ragas_llm),
        ),
        Parallel(
            CosineSimilarityBuilder(
                property_name="summary_embedding",
                new_property_name="summary_similarity",
                threshold=0.1,
            ),
            OverlapScoreBuilder(threshold=0.01),
        ),
    ]

    distribution = build_query_distribution(ragas_llm, weights or {})
    summary_str = ", ".join(f"{s.name}: {w:.0%}" for s, w in distribution)
    print(f"[DatasetGenerator-generate] Synthesizing {test_size} samples ({summary_str}) across {len(docs)} chunks...")

    generator = TestsetGenerator(
        llm=ragas_llm,
        embedding_model=ragas_embeddings,
        knowledge_graph=KnowledgeGraph(),
    )
    run_config = RunConfig(max_workers=4, max_retries=5, timeout=240)
    dataset = generator.generate_with_langchain_docs(
        documents=docs,
        testset_size=test_size,
        transforms=transforms,
        query_distribution=distribution,
        run_config=run_config,
    )

    # Clean flat schema for RAGAS
    samples = [
        {
            "id": f"sample-{idx + 1:02d}",
            "question": row.get("user_input") or row.get("question") or "",
            "ground_truth": row.get("reference") or row.get("ground_truth") or "",
            "ground_truth_contexts": list(row.get("reference_contexts") or []),
            "synthesizer": row.get("synthesizer_name") or "",
        }
        for idx, row in dataset.to_pandas().iterrows()
    ]

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(samples, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"[DatasetGenerator-save] Saved {len(samples)} samples to {output_path.name}")
    return samples
# endregion

In [ ]:
# region ExecGenerator
# Set EXECUTE_GENERATOR to True to trigger OpenRouter testset generation
EXECUTE_GENERATOR = False

gen_cfg = EVAL_CONFIG.get("generator", {})
if EXECUTE_GENERATOR:
    generated_samples = generate_eval_dataset_from_docs(
        doc_path=CURRENT_DIR / gen_cfg.get("doc_path", "0.doc.json"),
        output_path=CURRENT_DIR / gen_cfg.get("dataset_path", "1.dataset.json"),
        test_size=gen_cfg.get("test_size", 30),
        max_chunks=gen_cfg.get("max_chunks", 20),
        weights=gen_cfg.get("weights"),
    )
else:
    print(f"[DatasetGenerator-run] Generation skipped. Set EXECUTE_GENERATOR = True to re-synthesize test set from {DOCS_FILE.name}.")
# endregion

## Step 2: Query LangGraph Agent (Direct or Server)

Query the LangGraph RAG agent to collect generated responses and retrieved document contexts.
Settings are loaded from `eval_config.json["query"]`.

### Execution Modes:
- **`direct` (Recommended)**: Imports `agent_graph` from `src.agent_flow.graph` and executes directly in-process via `ainvoke()`. Fast, self-contained, no external server required.
- **`http`**: Sends requests to a running LangGraph Server (`http://localhost:2024/runs/wait`) via HTTP.

In [ ]:
# region Fetcher
async def fetch_single_langgraph_response(
    query: str,
    mode: str = "direct",
    endpoint_url: str = "http://localhost:2024",
    client: httpx.AsyncClient | None = None,
) -> tuple[str, list[str]]:
    """Fetches answer and retrieved contexts from LangGraph agent."""
    if mode == "direct":
        from src.agent_flow.graph import agent_graph

        state = await agent_graph.ainvoke({"query": query})
        answer = state.get("draft_response") or ""
        # Extract retrieved / ranked docs from LangGraph AgentState
        docs = state.get("ranked_docs") or state.get("retrieved_docs") or []
        contexts = [
            doc.page_content if hasattr(doc, "page_content") else str(doc)
            for doc in docs
        ]
        return answer, contexts

    elif mode == "http":
        if client is None:
            raise ValueError("[EvalRunner-fetch] AsyncClient must be provided for HTTP mode.")
        url = f"{endpoint_url.rstrip('/')}/runs/wait"
        payload = {"assistant_id": "agent", "input": {"query": query}}
        resp = await client.post(url, json=payload)
        resp.raise_for_status()
        data = resp.json()
        answer = data.get("draft_response") or ""
        raw_docs = data.get("ranked_docs") or data.get("retrieved_docs") or []
        contexts = [
            d.get("page_content", "") if isinstance(d, dict) else str(d)
            for d in raw_docs
        ]
        return answer, contexts

    else:
        raise ValueError(f"Unknown mode: {mode}")


async def fetch_all_langgraph_responses(
    dataset_path: Path,
    mode: str = "direct",
    endpoint_url: str = "http://localhost:2024",
    max_samples: int | None = None,
    timeout: float = 90.0,
    max_retries: int = 3,
) -> list[dict]:
    """Queries the LangGraph agent for each test sample in dataset."""
    if not dataset_path.exists():
        raise FileNotFoundError(f"[EvalRunner-fetch] Dataset file not found: {dataset_path}")

    dataset = json.loads(dataset_path.read_text(encoding="utf-8"))
    if max_samples:
        dataset = dataset[:max_samples]

    print(f"[EvalRunner-fetch] Running LangGraph ({mode} mode) across {len(dataset)} questions...")
    samples = []

    async with httpx.AsyncClient(timeout=timeout) as client:
        for idx, item in enumerate(dataset, 1):
            q = (item.get("question") or item.get("inputs", {}).get("query", "")).strip()
            if not q:
                continue

            print(f"[EvalRunner-fetch] [{idx:02d}/{len(dataset)}] Query: '{q[:60]}...'")
            answer, contexts = "Error executing query.", ["Error"]

            for attempt in range(1, max_retries + 1):
                try:
                    answer, contexts = await fetch_single_langgraph_response(
                        query=q,
                        mode=mode,
                        endpoint_url=endpoint_url,
                        client=client if mode == "http" else None,
                    )
                    break
                except Exception as err:
                    print(f"[EvalRunner-fetch] Attempt {attempt}/{max_retries} failed for '{q[:40]}': {err}")
                    if attempt < max_retries:
                        await asyncio.sleep(2.0 * attempt)

            gt_contexts = (
                item.get("ground_truth_contexts")
                or item.get("metadata", {}).get("ground_truth_contexts")
                or []
            )
            expected_answer = (
                item.get("ground_truth")
                or item.get("outputs", {}).get("ground_truth")
                or item.get("right_answer", "")
            )

            samples.append(
                {
                    "question": q,
                    "answer": answer,
                    "contexts": contexts,
                    "ground_truth": expected_answer,
                    "reference_contexts": gt_contexts,
                }
            )

    return samples
# endregion

In [ ]:
# region ExecFetch
# Execution configuration from eval_config.json:
EXECUTE_QUERY = False

query_cfg = EVAL_CONFIG.get("query", {})
QUERY_MODE = query_cfg.get("mode", "direct")  # Options: "direct" (in-process) or "http"
LANGGRAPH_ENDPOINT = query_cfg.get("endpoint_url", "http://localhost:2024")
MAX_SAMPLES = query_cfg.get("max_samples", None)
QUERY_TIMEOUT = float(query_cfg.get("timeout", 90.0))
QUERY_RETRIES = int(query_cfg.get("max_retries", 3))

rag_samples = []

if EXECUTE_QUERY:
    # In Jupyter, top-level await can also be used directly:
    # rag_samples = await fetch_all_langgraph_responses(DATASET_FILE, mode=QUERY_MODE)
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)

    run_coro = fetch_all_langgraph_responses(
        dataset_path=DATASET_FILE,
        mode=QUERY_MODE,
        endpoint_url=LANGGRAPH_ENDPOINT,
        max_samples=MAX_SAMPLES,
        timeout=QUERY_TIMEOUT,
        max_retries=QUERY_RETRIES,
    )

    if loop.is_running():
        import nest_asyncio
        nest_asyncio.apply()
        rag_samples = loop.run_until_complete(run_coro)
    else:
        rag_samples = loop.run_until_complete(run_coro)

    print(f"[EvalRunner-fetch] Collected {len(rag_samples)} responses from LangGraph agent.")
else:
    print("[EvalRunner-fetch] Query execution skipped. Set EXECUTE_QUERY = True to run queries.")
# endregion

## Step 3: Compute RAGAS Metrics

Evaluate retrieved contexts and answers against ground truth references using RAGAS:
- **Faithfulness**: Is the response grounded in the retrieved context? (Target: `>= 0.80`)
- **Answer Relevancy**: Is the response directly addressing the user inquiry? (Target: `>= 0.75`)
- **Context Precision**: How high is the ranking of relevant chunks in retrieval? (Target: `>= 0.85`)
- **Context Recall**: Did retrieval cover all facts from the ground truth answer? (Target: `>= 0.80`)

Outputs are written to timestamped CSV files (`2.run_eval_*.csv`).

In [ ]:
# region Evaluation
def run_ragas_evaluation(
    samples: list[dict],
    output_dir: Path,
    output_prefix: str = "2.run_eval_",
    max_workers: int = 2,
    timeout: int = 240,
    max_retries: int = 5,
) -> pd.DataFrame:
    """Executes RAGAS evaluation on collected samples and exports results to CSV."""
    if not samples:
        raise ValueError("[EvalRunner-eval] No samples provided for evaluation.")

    eval_llm, eval_embeddings = get_eval_models()
    eval_dataset = Dataset.from_dict(
        {
            "question": [s["question"] for s in samples],
            "answer": [s["answer"] for s in samples],
            "contexts": [s["contexts"] for s in samples],
            "ground_truth": [s["ground_truth"] for s in samples],
            "reference_contexts": [list(s.get("reference_contexts") or []) for s in samples],
        }
    )

    metrics = [faithfulness, answer_relevancy, context_precision, context_recall]
    print(f"[EvalRunner-eval] Evaluating {len(samples)} samples with RAGAS metrics...")
    
    result = evaluate(
        dataset=eval_dataset,
        metrics=metrics,
        llm=eval_llm,
        embeddings=eval_embeddings,
        run_config=RunConfig(max_workers=max_workers, max_retries=max_retries, timeout=timeout),
    )

    df: pd.DataFrame = result.to_pandas()
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = output_dir / f"{output_prefix}{timestamp}.csv"

    avg_row = {col: "-" for col in df.columns}
    avg_row["user_input"] = "[AVERAGE SUMMARY]"
    for m in metrics:
        if m.name in df.columns:
            avg_row[m.name] = round(float(df[m.name].mean()), 4)

    df_out = pd.concat([pd.DataFrame([avg_row]), df], ignore_index=True)
    df_out.to_csv(csv_path, index=False)

    print(f"\n[EvalRunner-save] CSV saved to: {csv_path}")
    print("\n### Mean Evaluation Scores:")
    for m in metrics:
        if m.name in avg_row:
            print(f"- {m.name:20s}: {avg_row[m.name]}")

    return df_out
# endregion

In [ ]:
# region ExecEval
# Set EXECUTE_EVAL to True to run RAGAS benchmark on collected samples
EXECUTE_EVAL = False

eval_cfg = EVAL_CONFIG.get("evaluation", {})
output_dir = CURRENT_DIR / eval_cfg.get("output_dir", ".")
output_prefix = eval_cfg.get("output_prefix", "2.run_eval_")

if EXECUTE_EVAL and rag_samples:
    eval_results_df = run_ragas_evaluation(
        samples=rag_samples,
        output_dir=output_dir,
        output_prefix=output_prefix,
        max_workers=eval_cfg.get("max_workers", 2),
        timeout=eval_cfg.get("timeout", 240),
        max_retries=eval_cfg.get("max_retries", 5),
    )
elif EXECUTE_EVAL and not rag_samples:
    print("[EvalRunner-eval] Cannot run evaluation: rag_samples is empty. Run Step 2 first.")
else:
    print("[EvalRunner-eval] Evaluation execution skipped. Set EXECUTE_EVAL = True to execute.")
# endregion

## Step 4: Benchmark Evolution, Visualization & Journal Synchronization

Track and report evaluation results:
1. **Journal Synchronization**: Optionally append the latest evaluation CSV summary row to `JOURNAL.md`.
2. **Historical Trend Analysis**: Load all historical benchmark CSV logs (`2.run_eval_*.csv`) and inspect summary tables.
3. **Trajectory Plotting**: Compare metric trajectories against targets defined in `eval_config.json["visualization"]`.

In [ ]:
# region Tracking
def sync_latest_eval_to_journal(
    eval_dir: Path,
    milestone_name: str,
    journal_path: Path | None = None,
) -> bool:
    """Appends the latest evaluation CSV summary row to JOURNAL.md."""
    if journal_path is None:
        j_rel = EVAL_CONFIG.get("journal", {}).get("journal_path", "JOURNAL.md")
        journal_path = eval_dir / j_rel

    if not journal_path.exists():
        print(f"[Journal-sync] Journal file not found: {journal_path}")
        return False

    csv_pattern = EVAL_CONFIG.get("visualization", {}).get("csv_pattern", "2.run_eval_*.csv")
    csv_files = sorted(eval_dir.glob(csv_pattern))
    if not csv_files:
        print(f"[Journal-sync] No evaluation CSV files found in {eval_dir}")
        return False

    latest_csv = csv_files[-1]
    match = re.search(r"2\.run_eval_(\d{8})_(\d{6})\.csv", latest_csv.name)
    if not match:
        print(f"[Journal-sync] Latest file format unexpected: {latest_csv.name}")
        return False

    date_str = f"{match.group(1)[:4]}-{match.group(1)[4:6]}-{match.group(1)[6:]}"
    time_str = f"{match.group(2)[:2]}:{match.group(2)[2:4]}:{match.group(2)[4:]}"

    df = pd.read_csv(latest_csv)
    summary_mask = df["user_input"].astype(str).str.contains("AVERAGE", case=False, na=False)
    summary_row = df[summary_mask].iloc[0] if summary_mask.any() else df[["faithfulness", "answer_relevancy", "context_precision", "context_recall"]].mean()

    f = float(summary_row.get("faithfulness", 0.0))
    ar = float(summary_row.get("answer_relevancy", 0.0))
    cp = float(summary_row.get("context_precision", 0.0))
    cr = float(summary_row.get("context_recall", 0.0))

    new_row = f"| **{date_str}** (`{time_str}`) | {milestone_name} | `{f:.4f}` | `{ar:.4f}` | `{cp:.4f}` | `{cr:.4f}` |"
    print(f"[Journal-sync] Generated Row:\n{new_row}")

    content = journal_path.read_text(encoding="utf-8")
    lines = content.splitlines()

    table_indices = [idx for idx, line in enumerate(lines) if line.strip().startswith("|") and ("Faithfulness" in line or "`" in line or ":---" in line)]
    if not table_indices:
        print("[Journal-sync] Could not locate benchmark summary table in journal.")
        return False

    last_table_idx = max(table_indices)
    lines.insert(last_table_idx + 1, new_row)
    journal_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"[Journal-sync] Successfully appended latest run to {journal_path.name}")
    return True


def load_benchmark_history(eval_dir: Path, csv_pattern: str = "2.run_eval_*.csv") -> pd.DataFrame:
    """Parses all historical evaluation CSV files and extracts summary rows."""
    csv_files = sorted(eval_dir.glob(csv_pattern))
    if not csv_files:
        print(f"[Visualization-load] No CSV benchmark files matching '{csv_pattern}' found in {eval_dir}")
        return pd.DataFrame()

    records = []
    metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

    for file_path in csv_files:
        match = re.search(r"2\.run_eval_(\d{8})_(\d{6})\.csv", file_path.name)
        if match:
            date_str = f"{match.group(1)[:4]}-{match.group(1)[4:6]}-{match.group(1)[6:]}"
            time_str = f"{match.group(2)[:2]}:{match.group(2)[2:4]}:{match.group(2)[4:]}"
            label = f"{date_str} {time_str}"
        else:
            label = file_path.stem.replace("2.run_eval_", "")

        try:
            df = pd.read_csv(file_path)
            summary_mask = df["user_input"].astype(str).str.contains("AVERAGE", case=False, na=False)
            if summary_mask.any():
                summary_row = df[summary_mask].iloc[0]
            else:
                summary_row = df[metric_cols].mean()

            entry = {"Run": label, "File": file_path.name}
            for col in metric_cols:
                val = summary_row.get(col, None)
                try:
                    entry[col] = round(float(val), 4)
                except (ValueError, TypeError):
                    entry[col] = None
            records.append(entry)
        except Exception as err:
            print(f"[Visualization-load] Error parsing {file_path.name}: {err}")

    return pd.DataFrame(records)


def plot_benchmark_trends(history_df: pd.DataFrame, targets: dict[str, float] | None = None):
    """Renders multi-line trajectory charts across historical benchmark runs."""
    if history_df.empty:
        print("[Visualization-plot] No benchmark data available to plot.")
        return

    metric_targets = targets or {
        "faithfulness": 0.80,
        "answer_relevancy": 0.75,
        "context_precision": 0.85,
        "context_recall": 0.80,
    }
    metric_colors = {
        "faithfulness": "#2563eb",
        "answer_relevancy": "#059669",
        "context_precision": "#d97706",
        "context_recall": "#9333ea",
    }

    fig, ax = plt.subplots(figsize=(11, 5.5), dpi=100)
    x_labels = history_df["Run"].tolist()
    x_indices = list(range(len(x_labels)))

    for metric, target in metric_targets.items():
        if metric in history_df.columns:
            values = history_df[metric].tolist()
            color = metric_colors.get(metric, "#475569")
            ax.plot(
                x_indices,
                values,
                marker="o",
                linewidth=2.2,
                label=f"{metric.replace('_', ' ').title()} (Target: {target:.2f})",
                color=color,
            )
            ax.axhline(y=target, color=color, linestyle=":", alpha=0.4)

    ax.set_title("RAG Metric Progression Across Benchmark Runs", fontsize=14, fontweight="bold", pad=14)
    ax.set_xlabel("Benchmark Run Timestamp", fontsize=11, labelpad=10)
    ax.set_ylabel("Score (0.0 - 1.0)", fontsize=11, labelpad=10)
    ax.set_xticks(x_indices)
    ax.set_xticklabels(x_labels, rotation=25, ha="right", fontsize=9)
    ax.set_ylim(0.2, 1.05)
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(loc="lower right", framealpha=0.9, fontsize=9)
    plt.tight_layout()
    plt.show()


# --- 1. Optional Journal Sync ---
EXECUTE_JOURNAL_SYNC = False
journal_cfg = EVAL_CONFIG.get("journal", {})
MILESTONE_TITLE = journal_cfg.get("milestone_title", "[LangGraph Agent] Native Graph Benchmark Run")

if EXECUTE_JOURNAL_SYNC:
    sync_latest_eval_to_journal(
        eval_dir=CURRENT_DIR,
        milestone_name=MILESTONE_TITLE,
    )
else:
    print("[Journal-sync] Sync skipped. Set EXECUTE_JOURNAL_SYNC = True when ready to log run.")

# --- 2. Historical Benchmark Summary & Visualization ---
viz_cfg = EVAL_CONFIG.get("visualization", {})
history_df = load_benchmark_history(CURRENT_DIR, csv_pattern=viz_cfg.get("csv_pattern", "2.run_eval_*.csv"))
print("### Historical Benchmark Summary Table:")
print(history_df.to_string(index=False))
print()
plot_benchmark_trends(history_df, targets=viz_cfg.get("targets"))
# endregion